# Profile Antibiotic Vocabulary



Profile antibiotic vocabulary and candidate reportable pairs

The first eligible culture-organism has already been selected
independently of antibiotic testing and susceptibility result.

This analysis evaluates which antibiotics were tested for the
selected records and measures usable categorized-result
coverage before a reviewed organism-antibiotic reporting grid
is defined.

Categorized results are source categories:
Susceptible, Resistant, Intermediate

Inconclusive, Synergism, missing, and conflicting results are
counted separately and are not included in the categorized
denominator.

1. Antibiotic vocabulary represented among selected
first culture-organism records.

In [1]:
%sql
select
    r.antibiotic,
    count(*) as endpoint_groups,
    count(distinct f.culture_organism_key) as culture_organisms,
    count(distinct f.patient_key) as patients

from workspace.micro_dev.int_first_culture_organism f

inner join workspace.micro_dev.int_susceptibility_reconciliation r
    on f.culture_organism_key = r.culture_organism_key

where r.antibiotic is not null

group by r.antibiotic

order by endpoint_groups desc, r.antibiotic;

,antibiotic,endpoint_groups,culture_organisms,patients
0,Gentamicin,64903,64903,54570
1,Trimethoprim/Sulfamethoxazole,61827,61827,53097
2,Ciprofloxacin,61015,61015,51538
3,Levofloxacin,56934,56934,47532
4,Ampicillin,56095,56095,48657
5,Piperacillin/Tazobactam,52991,52991,45693
6,Cefazolin,52252,52252,45948
7,Nitrofurantoin,50696,50696,44596
8,Ceftriaxone,43545,43545,38361
9,Meropenem,40392,40392,35184


2. Blood-cohort organism-antibiotic coverage.

eligible_count:
first eligible culture-organisms for the organism

categorized_count:
Susceptible + Resistant + Intermediate

categorized_result_coverage_pct:
categorized_count / eligible_count

endpoint_groups counts all records where the antibiotic
appears, including missing/conflicting/other categories.

untested_count represents eligible records with no endpoint
for that antibiotic.

In [2]:
%sql
with eligible as (

    select
        cohort_id,
        organism_standardized,
        count(*) as eligible_count

    from workspace.micro_dev.int_first_culture_organism

    group by
        cohort_id,
        organism_standardized

),

endpoints as (

    select
        f.cohort_id,
        f.organism_standardized,
        r.antibiotic,

        count(*) as endpoint_groups,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Resistant',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as categorized_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Susceptible'
                    then 1
                else 0
            end
        ) as susceptible_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Intermediate'
                    then 1
                else 0
            end
        ) as intermediate_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Inconclusive'
                    then 1
                else 0
            end
        ) as inconclusive_count,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility = 'Synergism'
                    then 1
                else 0
            end
        ) as synergism_count,

        sum(
            case
                when r.resolution_status = 'missing'
                    then 1
                else 0
            end
        ) as missing_result_count,

        sum(
            case
                when r.resolution_status = 'conflict'
                    then 1
                else 0
            end
        ) as conflict_result_count

    from workspace.micro_dev.int_first_culture_organism f

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on f.culture_organism_key = r.culture_organism_key

    where r.antibiotic is not null

    group by
        f.cohort_id,
        f.organism_standardized,
        r.antibiotic

)

select
    e.organism_standardized,
    p.antibiotic,

    e.eligible_count,
    p.endpoint_groups,

    e.eligible_count - p.endpoint_groups
        as untested_count,

    p.categorized_count,
    p.susceptible_count,
    p.resistant_count,
    p.intermediate_count,

    p.inconclusive_count,
    p.synergism_count,
    p.missing_result_count,
    p.conflict_result_count,

    round(
        100.0 * p.categorized_count / e.eligible_count,
        1
    ) as categorized_result_coverage_pct

from endpoints p

inner join eligible e
    on p.cohort_id = e.cohort_id
   and p.organism_standardized = e.organism_standardized

where p.cohort_id = 'blood'

order by
    e.organism_standardized,
    p.categorized_count desc,
    p.antibiotic;

,organism_standardized,antibiotic,eligible_count,endpoint_groups,untested_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count,categorized_result_coverage_pct
0,Enterobacter cloacae complex,Gentamicin,243,243,0,243,238,5,0,0,0,0,0,100.0
1,Enterobacter cloacae complex,Meropenem,243,243,0,243,242,1,0,0,0,0,0,100.0
2,Enterobacter cloacae complex,Trimethoprim/Sulfamethoxazole,243,243,0,243,225,18,0,0,0,0,0,100.0
3,Enterobacter cloacae complex,Ciprofloxacin,243,243,0,242,230,9,3,0,0,0,1,99.6
4,Enterobacter cloacae complex,Ertapenem,243,229,14,229,210,10,9,0,0,0,0,94.2
5,Enterobacter cloacae complex,Ceftriaxone,243,243,0,226,185,38,3,17,0,0,0,93.0
6,Enterobacter cloacae complex,Piperacillin/Tazobactam,243,243,0,225,201,17,7,17,0,0,1,92.6
7,Enterobacter cloacae complex,Ceftazidime,243,242,1,224,192,29,3,17,0,0,1,92.2
8,Enterobacter cloacae complex,Cefoxitin,243,223,20,223,0,223,0,0,0,0,0,91.8
9,Enterobacter cloacae complex,Levofloxacin,243,223,20,223,214,6,3,0,0,0,0,91.8


3. Top 10 candidate antibiotics per blood organism based
only on categorized-result count.

This is a profiling aid, not an automatic rule for inclusion.
Final organism-antibiotic pairs will be explicitly reviewed.

In [3]:
%sql
with eligible as (

    select
        cohort_id,
        organism_standardized,
        count(*) as eligible_count

    from workspace.micro_dev.int_first_culture_organism

    where cohort_id = 'blood'

    group by
        cohort_id,
        organism_standardized

),

pair_counts as (

    select
        f.organism_standardized,
        r.antibiotic,

        count(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Resistant',
                     'Intermediate'
                 )
                    then 1
            end
        ) as categorized_count

    from workspace.micro_dev.int_first_culture_organism f

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on f.culture_organism_key = r.culture_organism_key

    where f.cohort_id = 'blood'
      and r.antibiotic is not null

    group by
        f.organism_standardized,
        r.antibiotic

),

ranked as (

    select
        p.organism_standardized,
        p.antibiotic,
        e.eligible_count,
        p.categorized_count,

        round(
            100.0 * p.categorized_count / e.eligible_count,
            1
        ) as categorized_result_coverage_pct,

        row_number() over (
            partition by p.organism_standardized
            order by
                p.categorized_count desc,
                p.antibiotic
        ) as antibiotic_rank

    from pair_counts p

    inner join eligible e
        on p.organism_standardized = e.organism_standardized

)

select *
from ranked
where antibiotic_rank <= 10

order by
    organism_standardized,
    antibiotic_rank;

,organism_standardized,antibiotic,eligible_count,categorized_count,categorized_result_coverage_pct,antibiotic_rank
0,Enterobacter cloacae complex,Gentamicin,243,243,100.0,1
1,Enterobacter cloacae complex,Meropenem,243,243,100.0,2
2,Enterobacter cloacae complex,Trimethoprim/Sulfamethoxazole,243,243,100.0,3
3,Enterobacter cloacae complex,Ciprofloxacin,243,242,99.6,4
4,Enterobacter cloacae complex,Ertapenem,243,229,94.2,5
5,Enterobacter cloacae complex,Ceftriaxone,243,226,93.0,6
6,Enterobacter cloacae complex,Piperacillin/Tazobactam,243,225,92.6,7
7,Enterobacter cloacae complex,Ceftazidime,243,224,92.2,8
8,Enterobacter cloacae complex,Cefoxitin,243,223,91.8,9
9,Enterobacter cloacae complex,Levofloxacin,243,223,91.8,10


4. Detect possible formatting variants in antibiotic labels.

Expected result may be zero rows. Any matches require manual
review before terminology mappings are introduced.

In [4]:
%sql
select
    lower(trim(antibiotic)) as normalized_label,
    count(distinct antibiotic) as source_label_count,
    array_sort(collect_set(antibiotic)) as source_labels

from workspace.micro_dev.int_susceptibility_reconciliation

where antibiotic is not null

group by lower(trim(antibiotic))

having count(distinct antibiotic) > 1

order by source_label_count desc, normalized_label;

,normalized_label,source_label_count,source_labels


## Expanded antibiotic terminology audit

The preceding analyses use source antibiotic labels exactly as stored in
ARMD. Before those labels are used in phenotype rules, the vocabulary
must be checked for terminology variants.

The earlier comparison normalized only capitalization and surrounding
whitespace. This expanded audit also removes spaces, slashes, hyphens,
parentheses, and other punctuation when comparing labels.

The audit has three parts:

1. review the complete source antibiotic vocabulary and frequency;
2. identify multiple source labels that collapse to the same
   punctuation-insensitive comparison key;
3. confirm that the exact antibiotics required by the accepted CRE,
   oxacillin-resistant *S. aureus*, VRE, and MDR *Pseudomonas
   aeruginosa* rules can be matched in the source vocabulary.

A shared comparison key identifies candidate formatting variants only.
It does not automatically establish that differently worded drugs,
abbreviations, or combination products are equivalent. Any candidate
mapping must be reviewed before it is used in a production model.

In [5]:
%sql
with vocabulary as (

    select
        antibiotic,

        lower(
            trim(antibiotic)
        ) as case_space_normalized_label,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as punctuation_insensitive_key,

        count(*) as endpoint_groups,

        count(
            distinct culture_organism_key
        ) as culture_organisms

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    antibiotic,
    case_space_normalized_label,
    punctuation_insensitive_key,
    endpoint_groups,
    culture_organisms

from vocabulary

order by antibiotic

,antibiotic,case_space_normalized_label,punctuation_insensitive_key,endpoint_groups,culture_organisms
0,Amikacin,amikacin,amikacin,68148,68148
1,Amoxicillin/Clavulanic Acid,amoxicillin/clavulanic acid,amoxicillinclavulanicacid,57645,57645
2,Ampicillin,ampicillin,ampicillin,93245,93245
3,Ampicillin/Sulbactam,ampicillin/sulbactam,ampicillinsulbactam,21826,21826
4,Aztreonam,aztreonam,aztreonam,24755,24755
5,Cefazolin,cefazolin,cefazolin,82603,82603
6,Cefepime,cefepime,cefepime,37779,37779
7,Cefiderocol,cefiderocol,cefiderocol,20,20
8,Cefotaxime,cefotaxime,cefotaxime,765,765
9,Cefotetan,cefotetan,cefotetan,667,667


### Complete vocabulary review

This table exposes the complete analytical antibiotic vocabulary rather
than only drugs represented among selected first cultures.

The normalized key is included solely to make formatting differences
easier to identify. The source antibiotic label remains the authoritative
value unless a reviewed mapping is introduced.

In [6]:
%sql
with vocabulary as (

    select
        antibiotic,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as punctuation_insensitive_key,

        count(*) as endpoint_groups

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    punctuation_insensitive_key,

    count(
        distinct antibiotic
    ) as source_label_count,

    array_sort(
        collect_set(antibiotic)
    ) as source_labels,

    sum(endpoint_groups)
        as endpoint_groups

from vocabulary

group by punctuation_insensitive_key

having count(distinct antibiotic) > 1

order by
    source_label_count desc,
    punctuation_insensitive_key

,punctuation_insensitive_key,source_label_count,source_labels,endpoint_groups


### Interpretation of candidate variants

A zero-row result means no source labels differ only by capitalization,
whitespace, punctuation, slashes, parentheses, or hyphens.

A nonzero result does not automatically mean the labels should be
combined. Each candidate group must be reviewed to confirm that it
represents a formatting variant rather than genuinely different
antimicrobial agents or formulations.

In [7]:
%sql
with expected_antibiotics as (

    select *
    from values

        -- CRE phenotype
        ('CRE', 'Ertapenem'),
        ('CRE', 'Imipenem'),
        ('CRE', 'Meropenem'),
        ('CRE', 'Meropenem/Vaborbactam'),
        ('CRE', 'Imipenem/Relebactam'),

        -- Oxacillin-resistant Staphylococcus aureus
        ('OXACILLIN_RESISTANT_S_AUREUS', 'Oxacillin'),
        ('OXACILLIN_RESISTANT_S_AUREUS', 'Cefoxitin'),

        -- Vancomycin-resistant Enterococcus
        ('VRE', 'Vancomycin'),

        -- MDR Pseudomonas aeruginosa:
        -- extended-spectrum cephalosporins
        ('MDR_P_AERUGINOSA', 'Cefepime'),
        ('MDR_P_AERUGINOSA', 'Ceftazidime'),
        ('MDR_P_AERUGINOSA', 'Ceftazidime/Avibactam'),
        ('MDR_P_AERUGINOSA', 'Ceftolozane/Tazobactam'),

        -- fluoroquinolones
        ('MDR_P_AERUGINOSA', 'Ciprofloxacin'),
        ('MDR_P_AERUGINOSA', 'Levofloxacin'),

        -- aminoglycosides
        ('MDR_P_AERUGINOSA', 'Amikacin'),
        ('MDR_P_AERUGINOSA', 'Tobramycin'),

        -- carbapenems
        ('MDR_P_AERUGINOSA', 'Imipenem'),
        ('MDR_P_AERUGINOSA', 'Meropenem'),
        ('MDR_P_AERUGINOSA', 'Imipenem/Relebactam'),

        -- antipseudomonal penicillin combination
        ('MDR_P_AERUGINOSA', 'Piperacillin/Tazobactam'),

        -- siderophore cephalosporin
        ('MDR_P_AERUGINOSA', 'Cefiderocol')

    as expected(
        phenotype_rule,
        expected_antibiotic
    )

),

expected_normalized as (

    select
        phenotype_rule,
        expected_antibiotic,

        regexp_replace(
            lower(trim(expected_antibiotic)),
            '[^a-z0-9]',
            ''
        ) as comparison_key

    from expected_antibiotics

),

source_vocabulary as (

    select
        antibiotic,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as comparison_key,

        count(*) as endpoint_groups,

        count(
            distinct culture_organism_key
        ) as culture_organisms

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    e.phenotype_rule,
    e.expected_antibiotic,

    count(
        distinct v.antibiotic
    ) as matched_source_label_count,

    array_sort(
        collect_set(v.antibiotic)
    ) as matched_source_labels,

    coalesce(
        sum(v.endpoint_groups),
        0
    ) as endpoint_groups,

    coalesce(
        sum(v.culture_organisms),
        0
    ) as culture_organisms,

    case
        when count(distinct v.antibiotic) = 0
            then 'not_present'

        when count(distinct v.antibiotic) = 1
            then 'single_source_label'

        else 'multiple_candidate_labels'
    end as vocabulary_status

from expected_normalized e

left join source_vocabulary v
    on e.comparison_key = v.comparison_key

group by
    e.phenotype_rule,
    e.expected_antibiotic

order by
    e.phenotype_rule,
    e.expected_antibiotic

,phenotype_rule,expected_antibiotic,matched_source_label_count,matched_source_labels,endpoint_groups,culture_organisms,vocabulary_status
0,CRE,Ertapenem,1,[Ertapenem],57088,57088,single_source_label
1,CRE,Imipenem,1,[Imipenem],12171,12171,single_source_label
2,CRE,Imipenem/Relebactam,0,[],0,0,not_present
3,CRE,Meropenem,1,[Meropenem],70151,70151,single_source_label
4,CRE,Meropenem/Vaborbactam,1,[Meropenem/Vaborbactam],26,26,single_source_label
5,MDR_P_AERUGINOSA,Amikacin,1,[Amikacin],68148,68148,single_source_label
6,MDR_P_AERUGINOSA,Cefepime,1,[Cefepime],37779,37779,single_source_label
7,MDR_P_AERUGINOSA,Cefiderocol,1,[Cefiderocol],20,20,single_source_label
8,MDR_P_AERUGINOSA,Ceftazidime,1,[Ceftazidime],68473,68473,single_source_label
9,MDR_P_AERUGINOSA,Ceftazidime/Avibactam,1,[Ceftazidime/Avibactam],1488,1488,single_source_label


## Antibiotic terminology conclusion

The antibiotic vocabulary is considered ready for phenotype modeling
when:

- punctuation-insensitive comparison does not reveal unreviewed duplicate
  labels;
- every antibiotic used in a phenotype rule has either one reviewed
  source label or is explicitly documented as absent from ARMD;
- no phenotype rule relies on an unreviewed abbreviation or synonym.

An antibiotic marked `not_present` is not a data-quality error. It means
that the corresponding rule component cannot contribute evidence for
records in this ARMD release.

The project retains source antibiotic labels unless a reviewed,
version-controlled mapping becomes necessary.